In [0]:
create or replace view workspace.pj_sales.vw_rank_by_store_seller as (
  with
    aggregated_store_seller as (
      select
         tdst.store_id
        ,tdst.store_name
        ,tds.seller_id
        ,tds.seller_name
        ,count(distinct tfs.invoice_id)                                                  as sales_quantity
        ,round(sum(tfs.part_quantity * tfs.part_unit_price), 2)                          as billing
        ,round(sum(tfs.part_quantity * (tfs.part_unit_price - tfs.part_unit_cost)), 2)   as profit
        ,round(sum(tfs.part_quantity), 2)                                                as product_quantity
      from workspace.pj_sales.tb_fact_sale_gold tfs
      inner join workspace.pj_sales.tb_dim_seller_gold tds
        on tds._sk_seller = tfs._sk_seller
      inner join workspace.pj_sales.tb_dim_store_gold tdst
        on tdst._sk_store = tfs._sk_store
      where tfs.part_quantity > 0
      and   tfs.part_unit_price >= 0
      and   tfs.part_unit_cost >= 0
      group by
         tdst.store_id
        ,tdst.store_name
        ,tds.seller_id
        ,tds.seller_name
    )
  select
     asts.store_id
    ,asts.store_name
    ,asts.seller_id
    ,asts.seller_name
    ,asts.sales_quantity
    ,asts.billing
    ,asts.profit
    ,asts.product_quantity
    ,dense_rank() over (partition by asts.store_id order by asts.billing        desc)   as rank_by_billing
  from aggregated_store_seller asts
)